# AI Student Analytics & Risk Prediction — Complete Google Colab Notebook

This notebook trains the two main supervised-learning components used by the Streamlit college application:

1. **Grade Prediction** — predicts Grade from Gender, Age, Major, Attendance, Study Hours, Previous CGPA, Sleep Hours and Social Hours.
2. **Academic Risk Prediction** — predicts Low Risk / Medium Risk / High Risk from Major, Attendance, Previous CGPA, Study Hours, Sleep Hours and Social Hours.

It also:
- cleans the student dataset,
- performs EDA,
- creates targets if the supplied dataset does not already contain them,
- compares several models,
- saves the selected pipelines and encoders,
- saves evaluation metrics and feature importance,
- creates files that can be copied into the Streamlit project.

**Research note:** If Grade or Risk_Level is derived because the source dataset does not provide it, clearly state in the research paper that these are engineered targets.


In [ ]:
# 1. Install the exact versions used by the Streamlit project
!pip install -q --upgrade \
    pandas==2.2.3 \
    numpy==2.2.3 \
    scikit-learn==1.6.1 \
    xgboost==3.4.1 \
    joblib==1.6.0 \
    plotly==6.0.1 \
    openpyxl==3.1.5


In [ ]:
# 2. Verify environment
import sys, os, json, warnings
import pandas as pd
import numpy as np
import sklearn
import xgboost
import joblib

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Joblib:", joblib.__version__)


In [ ]:
# 3. Upload your CSV
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Please upload your student CSV file.")
CSV_PATH = csv_files[0]
print("Using:", CSV_PATH)


In [ ]:
# 4. Load and inspect
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
# 5. Standardize column names and resolve common variants
import re

def normalize_name(x):
    x = str(x).strip()
    x = re.sub(r"[^A-Za-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

df.columns = [normalize_name(c) for c in df.columns]

rename_map = {
    "Student_IDGenderAgeMajorAttendance_PctStudy_Hours_Per_DayPrevious_GPASleep_HoursSocial_Hours_WeekFinal_CGPA": None,
    "Previous_GPA": "Previous_CGPA",
    "Previous_GPA_": "Previous_CGPA",
    "Previous_CGPA": "Previous_CGPA",
    "Attendance": "Attendance_Pct",
    "Attendance_Percentage": "Attendance_Pct",
    "Study_Hours": "Study_Hours_Per_Day",
    "Study_Hours_Per_Day": "Study_Hours_Per_Day",
    "Sleep_Hours_Per_Day": "Sleep_Hours",
    "Social_Hours_Per_Week": "Social_Hours_Week",
    "Final_GPA": "Final_CGPA",
    "Final_GPA_": "Final_CGPA",
}

for old, new in list(rename_map.items()):
    if old in df.columns and new and old != new:
        df = df.rename(columns={old: new})

print(df.columns.tolist())


In [ ]:
# 6. Validate required source columns
required = [
    "Student_ID", "Gender", "Age", "Major",
    "Attendance_Pct", "Study_Hours_Per_Day",
    "Previous_CGPA", "Sleep_Hours",
    "Social_Hours_Week", "Final_CGPA"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Your CSV should contain the student fields used by this project."
    )

# Convert numeric columns safely
numeric_cols = [
    "Age", "Attendance_Pct", "Study_Hours_Per_Day",
    "Previous_CGPA", "Sleep_Hours", "Social_Hours_Week", "Final_CGPA"
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=required).copy()

# sensible clipping
df["Attendance_Pct"] = df["Attendance_Pct"].clip(0, 100)
df["Previous_CGPA"] = df["Previous_CGPA"].clip(0, 4)
df["Final_CGPA"] = df["Final_CGPA"].clip(0, 4)
df["Study_Hours_Per_Day"] = df["Study_Hours_Per_Day"].clip(0, 24)
df["Sleep_Hours"] = df["Sleep_Hours"].clip(0, 24)
df["Social_Hours_Week"] = df["Social_Hours_Week"].clip(0, 168)

print("Clean shape:", df.shape)
display(df.describe(include="all").T)


## 7. Exploratory Data Analysis

The following plots are useful for the research presentation:
- Attendance vs Final CGPA
- Study Hours vs Final CGPA
- Previous CGPA vs Final CGPA
- Final CGPA distribution
- Students by Major

Use these to explain relationships before training models.


In [ ]:
# 7. EDA
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=df, x="Attendance_Pct", y="Final_CGPA", hue="Major", ax=ax)
ax.set_title("Attendance vs Final CGPA")
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=df, x="Study_Hours_Per_Day", y="Final_CGPA", hue="Major", ax=ax)
ax.set_title("Study Hours vs Final CGPA")
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(data=df, x="Final_CGPA", kde=True, ax=ax)
ax.set_title("Final CGPA Distribution")
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
df["Major"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Students by Major")
ax.set_ylabel("Students")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# 8. Create Grade target if the dataset does not already contain one
# You can adjust these bands in your research methodology if your institution uses different grading rules.

def cgpa_to_grade(cgpa):
    if cgpa >= 3.70:
        return "A+"
    elif cgpa >= 3.30:
        return "A"
    elif cgpa >= 3.00:
        return "B"
    elif cgpa >= 2.50:
        return "C"
    elif cgpa >= 2.00:
        return "D"
    return "F"

if "Grade" not in df.columns:
    df["Grade"] = df["Final_CGPA"].apply(cgpa_to_grade)

print(df["Grade"].value_counts())


In [ ]:
# 9. Create Risk_Level target if it is not present
# This is an engineered academic-risk target based on observable academic indicators.
# It is deliberately transparent so it can be explained in a research presentation.

def risk_from_row(row):
    risk_points = 0

    if row["Attendance_Pct"] < 60:
        risk_points += 3
    elif row["Attendance_Pct"] < 75:
        risk_points += 1

    if row["Previous_CGPA"] < 2.0:
        risk_points += 3
    elif row["Previous_CGPA"] < 2.5:
        risk_points += 2
    elif row["Previous_CGPA"] < 3.0:
        risk_points += 1

    if row["Study_Hours_Per_Day"] < 1.5:
        risk_points += 2
    elif row["Study_Hours_Per_Day"] < 3:
        risk_points += 1

    if row["Final_CGPA"] < 2.0:
        risk_points += 2
    elif row["Final_CGPA"] < 2.5:
        risk_points += 1

    if risk_points >= 5:
        return "High Risk"
    elif risk_points >= 2:
        return "Medium Risk"
    return "Low Risk"

if "Risk_Level" not in df.columns:
    df["Risk_Level"] = df.apply(risk_from_row, axis=1)

print(df["Risk_Level"].value_counts())


In [ ]:
# 10. Correlation analysis
corr_cols = [
    "Attendance_Pct", "Study_Hours_Per_Day",
    "Previous_CGPA", "Sleep_Hours",
    "Social_Hours_Week", "Final_CGPA"
]
display(df[corr_cols].corr().round(2))


In [ ]:
# 11. Prepare models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier

grade_features = [
    "Gender", "Age", "Major", "Attendance_Pct",
    "Study_Hours_Per_Day", "Previous_CGPA",
    "Sleep_Hours", "Social_Hours_Week"
]

risk_features = [
    "Major", "Attendance_Pct", "Previous_CGPA",
    "Study_Hours_Per_Day", "Sleep_Hours", "Social_Hours_Week"
]

X_grade = df[grade_features].copy()
y_grade_text = df["Grade"].astype(str).copy()

X_risk = df[risk_features].copy()
y_risk_text = df["Risk_Level"].astype(str).copy()

grade_encoder = LabelEncoder()
risk_encoder = LabelEncoder()

y_grade = grade_encoder.fit_transform(y_grade_text)
y_risk = risk_encoder.fit_transform(y_risk_text)

grade_cat = ["Gender", "Major"]
grade_num = [c for c in grade_features if c not in grade_cat]

risk_cat = ["Major"]
risk_num = [c for c in risk_features if c not in risk_cat]

grade_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), grade_cat),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), grade_num)
])

risk_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), risk_cat),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), risk_num)
])

Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_grade, y_grade, test_size=0.20, random_state=42, stratify=y_grade
)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_risk, y_risk, test_size=0.20, random_state=42, stratify=y_risk
)

print("Grade classes:", list(grade_encoder.classes_))
print("Risk classes:", list(risk_encoder.classes_))


In [ ]:
# 12. Candidate models
grade_models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=2
    )
}

risk_models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=2
    )
}


In [ ]:
# 13. Evaluation helper
def evaluate_models(models, preprocessor, X_train, X_test, y_train, y_test):
    results = {}
    fitted = {}

    for name, estimator in models.items():
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", estimator)
        ])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        results[name] = {
            "Accuracy": round(accuracy_score(y_test, pred), 4),
            "Precision_weighted": round(
                precision_score(y_test, pred, average="weighted", zero_division=0), 4
            ),
            "Recall_weighted": round(
                recall_score(y_test, pred, average="weighted", zero_division=0), 4
            ),
            "F1_weighted": round(
                f1_score(y_test, pred, average="weighted", zero_division=0), 4
            )
        }
        fitted[name] = pipe

        print("\n" + "="*70)
        print(name)
        print(classification_report(y_test, pred, zero_division=0))

    return results, fitted


In [ ]:
# 14. Train and compare Grade models
grade_results, grade_fitted = evaluate_models(
    grade_models, grade_preprocessor,
    Xg_train, Xg_test, yg_train, yg_test
)

grade_results_df = pd.DataFrame(grade_results).T.sort_values("F1_weighted", ascending=False)
display(grade_results_df)


In [ ]:
# 15. Train and compare Risk models
risk_results, risk_fitted = evaluate_models(
    risk_models, risk_preprocessor,
    Xr_train, Xr_test, yr_train, yr_test
)

risk_results_df = pd.DataFrame(risk_results).T.sort_values("F1_weighted", ascending=False)
display(risk_results_df)


In [ ]:
# 16. Select deployment models by weighted F1
best_grade_name = grade_results_df["F1_weighted"].idxmax()
best_risk_name = risk_results_df["F1_weighted"].idxmax()

best_grade_model = grade_fitted[best_grade_name]
best_risk_model = risk_fitted[best_risk_name]

print("Best Grade model:", best_grade_name)
print("Best Risk model:", best_risk_name)


In [ ]:
# 17. Save outputs
from pathlib import Path
import joblib, json

OUT = Path("/content/streamlit_artifacts")
(OUT / "models").mkdir(parents=True, exist_ok=True)
(OUT / "data").mkdir(parents=True, exist_ok=True)

joblib.dump(best_grade_model, OUT / "models" / "grade_model.pkl")
joblib.dump(grade_encoder, OUT / "models" / "grade_encoder.pkl")

joblib.dump(best_risk_model, OUT / "models" / "risk_model.pkl")
joblib.dump(risk_encoder, OUT / "models" / "risk_encoder.pkl")

grade_results_df.to_csv(OUT / "models" / "grade_model_comparison.csv")
risk_results_df.to_csv(OUT / "models" / "risk_model_comparison.csv")

metrics = {
    "grade": grade_results,
    "risk": risk_results,
    "selected_models": {
        "grade": best_grade_name,
        "risk": best_risk_name
    }
}
with open(OUT / "models" / "model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# Save cleaned data and model-ready data
df.to_csv(OUT / "data" / "student_data_cleaned.csv", index=False)

print("Saved artifacts to:", OUT)


In [ ]:
# 18. Feature importance for tree-based deployment models when available
import numpy as np

def save_feature_importance(pipe, filename):
    model = pipe.named_steps["model"]
    pre = pipe.named_steps["preprocessor"]

    if not hasattr(model, "feature_importances_"):
        print("Selected model does not expose feature_importances_. Skipping.")
        return

    feature_names = pre.get_feature_names_out()
    importance = model.feature_importances_

    imp = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    }).sort_values("Importance", ascending=False)

    imp.to_csv(OUT / "models" / filename, index=False)
    display(imp.head(20))

save_feature_importance(best_grade_model, "grade_feature_importance.csv")
save_feature_importance(best_risk_model, "risk_feature_importance.csv")


In [ ]:
# 19. Test the exact inference inputs used by Streamlit
test_grade = pd.DataFrame([{
    "Gender": "Female",
    "Age": 21,
    "Major": df["Major"].mode()[0],
    "Attendance_Pct": 75.0,
    "Study_Hours_Per_Day": 4.0,
    "Previous_CGPA": 2.8,
    "Sleep_Hours": 7.0,
    "Social_Hours_Week": 8.0
}])

test_risk = pd.DataFrame([{
    "Major": df["Major"].mode()[0],
    "Attendance_Pct": 75.0,
    "Previous_CGPA": 2.8,
    "Study_Hours_Per_Day": 4.0,
    "Sleep_Hours": 7.0,
    "Social_Hours_Week": 8.0
}])

g_pred = best_grade_model.predict(test_grade)[0]
r_pred = best_risk_model.predict(test_risk)[0]

print("Test Grade:", grade_encoder.inverse_transform([g_pred])[0])
print("Test Risk:", risk_encoder.inverse_transform([r_pred])[0])


In [ ]:
# 20. Zip the generated artifacts for download
import shutil

zip_path = shutil.make_archive(
    "/content/AI_Student_Analytics_Streamlit_Artifacts",
    "zip",
    OUT
)

print("ZIP created:", zip_path)
from google.colab import files
files.download(zip_path)
